# Bias-correct the coarse SEAS5 forecast 

Same method as the earlier bias-correction notebook, with three changes:
1. **This notebook corrects:** temp (additive), rh (additive), wind speed (capped ratio), tmax / tmin (same offset as temp), rainfall, wind direction.
2. **The bias maps are filled** (nearest valid pixel) so no pixel is lost at the edges of a county. 
3. Output goes to a (`FORECASTED_WEATHER_BC_V2`).


In [ ]:
import os, re, glob, pickle, warnings, shutil, time, filecmp
import numpy as np, pandas as pd, xarray as xr
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.warp import reproject
from scipy import ndimage
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ============================== SETTINGS ==============================
COUNTIES = ["Kakamega", "Kilifi", "Nyeri"]
STATS    = ["mean", "p10", "p90"]
IN_DIR   = r"F:\VI_prediction\FORECASTED_WEATHER"            # original coarse forecast: <group>/<var>/<stat>/<County>_<var>_<stat>_<date>.tif
OUT_DIR  = r"F:\VI_prediction\FORECASTED_WEATHER_BC_V2"      # NEW folder: corrected temp/rh/wind + unchanged rain and wind_dir
ARCHIVE_DIR = r"F:\SEAS5_ARCHIVE\daily"                      # SEAS5_daily_<YYYY-MM>.nc
DATE_START, DATE_END = "2026-10-01", "2027-05-03"

START_MONTHS = [10]
YEARS = range(2020, 2027)
MIN_DAYS_MONTH = 10

HIST_ROOT = r"F:\VI_prediction\HISTORICAL WEATHER"
HIST_WX = {                                                   # no rain here: rain is not corrected
    "temp_max": os.path.join(HIST_ROOT, "ERA5_LAND_TEMP_fixed", "{county}", "tmax", "{county}_tmax_{year}{month:02d}*.tif"),
    "temp_min": os.path.join(HIST_ROOT, "ERA5_LAND_TEMP_fixed", "{county}", "tmin", "{county}_tmin_{year}{month:02d}*.tif"),
    "rh":       os.path.join(HIST_ROOT, "DAILY_RH", "RH_{county_upper}", "daily_tifs", "{county}_RH_{year}-{month:02d}-*.tif"),
    "wind":     os.path.join(HIST_ROOT, "WIND_DATA_PY_daily", "county_daily_direct", "{county}", "WIND_{county}_{year}_{month:02d}_*.tif"),
}
WIND_BAND    = 1
MIN_DAY_FRAC = 0.8
DEM_PATH = r"F:\VI_prediction\SRTM\{county}\{county_lower}_dem_clipped.tif"     # only for the elevation check

RATIO_LO, RATIO_HI = 0.4, 2.5        # caps on the wind ratio
APPLY_TO_TXN  = True                 # tmax / tmin get the same offset as the mean temperature
SKIP_EXISTING = False
PLOT_MONTH    = 11

# (folder group, variable, kind). kind "copy" = file copied unchanged
LAYERS = [("temperature", "temp", "temp"), ("humidity", "rh", "rh"), ("rainfall", "rain", "copy"),
          ("wind", "wind_speed", "wind"), ("wind", "wind_dir", "copy"),
          ("temperature", "tmax", "tx"), ("temperature", "tmin", "tx")]
VARS_S5 = {"temp": "temp_c", "rh": "rh_pct", "wind": "wind_speed"}     # names inside the archive files
VARS3   = ["temp", "rh", "wind"]
FC_MONTHS = [(p.year, p.month) for p in pd.period_range(DATE_START, DATE_END, freq="M")]
CACHE_DIR = os.path.join(OUT_DIR, "_bias_cache"); os.makedirs(CACHE_DIR, exist_ok=True)
print("forecast months:", FC_MONTHS)

## 1. Helpers

In [ ]:
def fill_nearest(a):
    m = np.isnan(a)
    if not m.any() or m.all():
        return a
    idx = ndimage.distance_transform_edt(m, return_distances=False, return_indices=True)
    return a[tuple(idx)]

def to_grid(a, src_tf, src_crs, G, resampling=Resampling.bilinear):
    dst = np.full(G["shape"], np.nan, dtype="float32")
    reproject(np.ascontiguousarray(a, dtype="float32"), dst, src_transform=src_tf, src_crs=src_crs, dst_transform=G["transform"],
              dst_crs=G["crs"], resampling=resampling, src_nodata=np.nan, dst_nodata=np.nan)
    return dst

in_path  = lambda c, g, v, s, d: os.path.join(IN_DIR,  g, v, s, f"{c}_{v}_{s}_{d}.tif")
out_path = lambda c, g, v, s, d: os.path.join(OUT_DIR, g, v, s, f"{c}_{v}_{s}_{d}.tif")

def list_dates(county, group, var, stat):
    found = {}
    for p in glob.glob(os.path.join(IN_DIR, group, var, stat, f"{county}_{var}_{stat}_*.tif")):
        m = re.search(r"_(\d{4}-\d{2}-\d{2})\.tif$", p)          # ignores *_stack.tif
        if m and DATE_START <= m.group(1) <= DATE_END:
            found[m.group(1)] = p
    return sorted(found)

def build_grid(county):
    dates = list_dates(county, "temperature", "temp", "mean")
    if not dates:
        raise FileNotFoundError(f"no temp/mean forecast files for {county} in {IN_DIR} between {DATE_START} and {DATE_END}")
    with rasterio.open(in_path(county, "temperature", "temp", "mean", dates[0])) as s:
        a = s.read(1, masked=True).astype("float32").filled(np.nan)
        return dict(transform=s.transform, crs=s.crs, shape=s.shape, valid=np.isfinite(a), n_dates=len(dates))

def _date_of(path):
    for mt in re.finditer(r"(?<!\d)(\d{4})[-_]?(\d{2})[-_]?(\d{2})(?!\d)", os.path.basename(path)):
        y, mo, d = map(int, mt.groups())
        if 2000 <= y <= 2100 and 1 <= mo <= 12 and 1 <= d <= 31:
            return y, mo, d
    return None

def daily_files(key, county, y, m):
    tpl = HIST_WX[key]
    files = sorted(f for f in glob.glob(tpl.format(county=county, county_lower=county.lower(), county_upper=county.upper(), year=y, month=m))
                   if f.lower().endswith(".tif"))
    out = {}
    for f in files:
        d = _date_of(f)
        if d and d[0] == y and d[1] == m:
            out[d[2]] = f
    return out

def fix_units(var, a):
    if var.startswith("temp") and np.nanmean(a) > 100:      # Kelvin -> degC
        a = a - 273.15
    if var == "rh" and np.nanmax(a) <= 1.5:                 # fraction -> %
        a = a * 100
    return a

def hist_month(county, key, y, m, days, G):
    files = daily_files(key, county, y, m)
    use = [files[d] for d in days if d in files]
    if len(use) < MIN_DAY_FRAC * len(days):
        return None
    var = "temp" if key.startswith("temp") else key
    acc = cnt = None
    for f in use:
        with rasterio.open(f) as s:
            b = WIND_BAND if key == "wind" else 1
            a = s.read(b, masked=True).astype("float32").filled(np.nan); tf, crs = s.transform, s.crs
        a = fix_units(var, a)
        ok = np.isfinite(a)
        acc = np.where(ok, a, 0) if acc is None else acc + np.where(ok, a, 0)
        cnt = ok.astype("float32") if cnt is None else cnt + ok
    mean = np.where(cnt > 0, acc / np.maximum(cnt, 1), np.nan).astype("float32")
    return to_grid(fill_nearest(mean), tf, crs, G)

def s5_fields(ym):
    p = os.path.join(ARCHIVE_DIR, f"SEAS5_daily_{ym}.nc")
    if not os.path.exists(p):
        return None
    with xr.open_dataset(p) as ds:
        ds = ds.sortby("latitude").sortby("longitude")
        lat, lon = ds["latitude"].values, ds["longitude"].values
        dx, dy = abs(float(lon[1] - lon[0])), abs(float(lat[1] - lat[0]))
        tf = Affine(dx, 0, float(lon.min()) - dx / 2, 0, -dy, float(lat.max()) + dy / 2)
        dates = pd.DatetimeIndex(ds["date"].values)
        em = {k: ds[v].mean("number").values for k, v in VARS_S5.items()}
    out = {}
    for (y, m), g in pd.Series(np.arange(len(dates)), index=dates).groupby([dates.year, dates.month]):
        idx = g.values
        if len(idx) < MIN_DAYS_MONTH:
            continue
        out[(int(y), int(m))] = ([int(dates[i].day) for i in idx],
                                 {k: np.nanmean(a[idx], axis=0)[::-1].astype("float32") for k, a in em.items()})
    return out, tf

def _hist_stamp(county):
    out = []
    for key in HIST_WX:
        for y in YEARS:
            for m in START_MONTHS + [mm for _, mm in FC_MONTHS]:
                fs = daily_files(key, county, y, m)
                out.append((key, y, m, len(fs),
                            max((os.path.getmtime(f) for f in fs.values()), default=0)))
    return tuple(out)

def build_records(county, G):
    """One record per (archive start month, valid month): S = archive ensemble mean, H = history, on the county's 0.05 deg grid."""
    cache = os.path.join(CACHE_DIR, f"{county}_records.pkl")
    sig = (tuple(START_MONTHS), tuple(YEARS), MIN_DAYS_MONTH, MIN_DAY_FRAC, tuple(G["shape"]),
           tuple(round(v, 6) for v in tuple(G["transform"])[:6]), tuple(sorted(HIST_WX.items())),
           _hist_stamp(county))
    if os.path.exists(cache):
        z = pickle.load(open(cache, "rb"))
        if z["sig"] == sig:
            print(f"{county}: {len(z['recs'])} records loaded from cache"); return z["recs"]
    recs, skipped, t0 = [], [], time.time()
    for y0 in YEARS:
        for m0 in START_MONTHS:
            ym = f"{y0}-{m0:02d}"
            r = s5_fields(ym)
            if r is None:
                continue
            flds, tf5 = r
            for (y, m), (days, f) in flds.items():
                H = {k: hist_month(county, k, y, m, days, G) for k in ("temp_max", "temp_min", "rh", "wind")}
                miss = [k for k, v in H.items() if v is None]
                if miss:
                    skipped.append(f"{ym}->{y}-{m:02d} ({', '.join(miss)})"); continue
                S = {k: to_grid(fill_nearest(a), tf5, "EPSG:4326", G) for k, a in f.items()}
                recs.append(dict(start=ym, y=y, m=m, n_days=len(days), S=S,
                                 H=dict(temp=(H["temp_max"] + H["temp_min"]) / 2, rh=H["rh"], wind=H["wind"])))
            print(f"  {county} archive {ym}: {len(recs)} records so far ({time.time() - t0:.0f}s)", flush=True)
    if skipped:
        print(f"  {county}: skipped {len(skipped)} because history was missing: {skipped[:6]}{' ...' if len(skipped) > 6 else ''}")
    pickle.dump(dict(sig=sig, recs=recs), open(cache, "wb"))
    return recs

## 2. Compare the archive with history
Fast if the first run's records are reused; otherwise this is the slow cell.

In [ ]:
GRIDS, RECS = {}, {}
for c in COUNTIES:
    GRIDS[c] = build_grid(c)
    print(f"{c}: forecast grid {GRIDS[c]['shape']} {GRIDS[c]['crs']} | {GRIDS[c]['n_dates']} forecast days | valid pixels {int(GRIDS[c]['valid'].sum())}")
    RECS[c] = build_records(c, GRIDS[c])
    if not RECS[c]:
        raise RuntimeError(f"{c}: no (archive, history) pairs found: check ARCHIVE_DIR, START_MONTHS, YEARS and HIST_WX")

## 3. Bias per calendar month
- **temp, rh:** `history - archive`, averaged over the years, per pixel (additive).
- **wind:** `sum(history) / sum(archive)`, capped to `[RATIO_LO, RATIO_HI]` (multiplicative).
- Every map is **gap-filled** with the nearest valid pixel, so no pixel can turn into NaN after correction.

In [ ]:
FLOOR = {"wind": 0.1}     # lower bound for the archive value in a ratio (m/s)

def compute_bias(recs, valid):
    bias = {v: {} for v in VARS3}; rows = []
    for m in sorted({r["m"] for r in recs}):
        rr = [r for r in recs if r["m"] == m]; n = len(rr)
        row = dict(month=m, n=n)
        for v in ("temp", "rh"):
            d = np.stack([r["H"][v] - r["S"][v] for r in rr])
            b = fill_nearest(d.mean(0).astype("float32"))
            bias[v][m] = np.where(np.isfinite(b), b, 0.0).astype("float32")
            cm = np.array([np.nanmean(x[valid]) for x in d])
            row[v] = cm.mean(); row[v + "_se"] = cm.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
        sH, sS = np.sum([r["H"]["wind"] for r in rr], 0), np.sum([r["S"]["wind"] for r in rr], 0)
        raw = (sH / np.maximum(sS, FLOOR["wind"] * n)).astype("float32")
        b = fill_nearest(np.clip(raw, RATIO_LO, RATIO_HI))
        bias["wind"][m] = np.where(np.isfinite(b), b, 1.0).astype("float32")
        row["wind_ratio"] = float(np.nanmean(bias["wind"][m][valid]))
        row["wind_capped"] = float(np.mean(((raw < RATIO_LO) | (raw > RATIO_HI))[valid]))
        rows.append(row)
    return bias, pd.DataFrame(rows)

BIAS, SUMMARY = {}, {}
for c in COUNTIES:
    BIAS[c], SUMMARY[c] = compute_bias(RECS[c], GRIDS[c]["valid"])
    print(f"\n=== {c}: bias to ADD (temp degC, rh %) or MULTIPLY BY (wind) | records: {len(RECS[c])} ===")
    print(SUMMARY[c].round(2).to_string(index=False))
    hit = SUMMARY[c][SUMMARY[c].wind_capped > 0.05]
    if len(hit): print(f"  note: the wind cap is hit on more than 5 % of the pixels in months {hit.month.tolist()}")

for c in COUNTIES:      # every bias map must be free of NaN
    bad = [(v, m) for v in VARS3 for m, a in BIAS[c][v].items() if not np.isfinite(a).all()]
    print(c, "bias maps still containing NaN:", bad)

## 4. Checks before applying anything
Bias maps for one month, bias vs elevation, and the leave-one-out test (error should fall for temp, rh and wind).

In [ ]:
for c in COUNTIES:
    G = GRIDS[c]; m = PLOT_MONTH if PLOT_MONTH in BIAS[c]["temp"] else sorted(BIAS[c]["temp"])[0]
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    for a, v, t, cm in zip(ax, VARS3, ["temp: add (degC)", "rh: add (%)", "wind: x factor"], ["coolwarm", "BrBG", "PuOr"]):
        arr = np.where(G["valid"], BIAS[c][v][m], np.nan)
        lim = np.nanmax(np.abs(arr)) if v in ("temp", "rh") else None
        im = a.imshow(arr, cmap=cm, vmin=(-lim if lim else None), vmax=(lim if lim else None))
        a.set_title(f"{c} {t} (month {m})", fontsize=9); a.axis("off"); plt.colorbar(im, ax=a, shrink=.8)
    plt.tight_layout(); plt.show()

In [ ]:
def elev_on_grid(county, G):
    out = np.full(G["shape"], np.nan, "float32")
    with rasterio.open(DEM_PATH.format(county=county, county_lower=county.lower())) as s:
        reproject(rasterio.band(s, 1), out, dst_transform=G["transform"], dst_crs=G["crs"], resampling=Resampling.average,
                  src_nodata=s.nodata if s.nodata is not None else -9999, dst_nodata=np.nan)
    return out

fig, ax = plt.subplots(1, len(COUNTIES), figsize=(4.6 * len(COUNTIES), 3.6), squeeze=False)
for k, c in enumerate(COUNTIES):
    try:
        E = elev_on_grid(c, GRIDS[c])
    except Exception as e:
        print(f"{c}: DEM not read ({e})"); continue
    B = np.mean([BIAS[c]["temp"][m] for m in BIAS[c]["temp"]], 0)
    ok = GRIDS[c]["valid"] & np.isfinite(E) & np.isfinite(B)
    slope, icpt = np.polyfit(E[ok], B[ok], 1); r = np.corrcoef(E[ok], B[ok])[0, 1]
    print(f"{c}: temperature bias vs elevation: slope {slope * 100:+.2f} degC per 100 m, correlation {r:+.2f}")
    ax[0, k].scatter(E[ok], B[ok], s=3, alpha=.4); ax[0, k].set_title(f"{c}: slope {slope * 100:+.2f} degC/100 m, r={r:+.2f}", fontsize=9)
    ax[0, k].set_xlabel("elevation of the 0.05 deg pixel (m)"); ax[0, k].set_ylabel("mean temp bias to add (degC)")
plt.tight_layout(); plt.show()

In [ ]:
def loo_check(recs, valid):
    rows = []
    for m in sorted({r["m"] for r in recs}):
        rr = [r for r in recs if r["m"] == m]
        if len(rr) < 3: continue
        for i, r in enumerate(rr):
            rest = rr[:i] + rr[i + 1:]; n = len(rest)
            for v in ("temp", "rh"):
                b = np.mean([x["H"][v] - x["S"][v] for x in rest], 0)
                rows.append((v, np.nanmean(np.abs(r["S"][v] - r["H"][v])[valid]), np.nanmean(np.abs(r["S"][v] + b - r["H"][v])[valid])))
            sH, sS = np.sum([x["H"]["wind"] for x in rest], 0), np.sum([x["S"]["wind"] for x in rest], 0)
            f = np.clip(sH / np.maximum(sS, FLOOR["wind"] * n), RATIO_LO, RATIO_HI)
            rows.append(("wind", np.nanmean(np.abs(r["S"]["wind"] - r["H"]["wind"])[valid]), np.nanmean(np.abs(r["S"]["wind"] * f - r["H"]["wind"])[valid])))
    d = pd.DataFrame(rows, columns=["var", "MAE_before", "MAE_after"]).groupby("var").mean()
    d["change_%"] = 100 * (d.MAE_after / d.MAE_before - 1)
    return d.loc[[v for v in VARS3 if v in d.index]]

print("Leave-one-out: mean absolute error of the monthly mean field. 'after' should be clearly lower than 'before'.")
for c in COUNTIES:
    print(f"\n{c}"); print(loo_check(RECS[c], GRIDS[c]["valid"]).round(2).to_string())

## 5. Write the new folder
- temp, rh, wind speed, tmax, tmin: corrected.
- rain and wind direction: **copied unchanged**.
Files are written to a temporary name first, so an interruption never leaves a corrupt `.tif`. Re-running skips files that already exist.

In [ ]:
def chrono_table(by_month, months):
    have = [k for k in months if k[1] in by_month]
    if not have: raise ValueError("no bias estimate for any forecast month")
    tab, borrowed = {}, []
    for k in months:
        if k[1] in by_month: tab[k] = by_month[k[1]]
        else:
            prev = [h for h in have if h < k]; src = prev[-1] if prev else have[0]
            tab[k] = by_month[src[1]]; borrowed.append((k, src))
    return tab, borrowed

def at_date(table, d):
    """Linear interpolation between the month mid-points (the 15th)."""
    keys = sorted(table); mids = [pd.Timestamp(y, m, 15) for y, m in keys]; t = pd.Timestamp(d)
    if t <= mids[0]: return table[keys[0]]
    if t >= mids[-1]: return table[keys[-1]]
    j = next(i for i, x in enumerate(mids) if x >= t)
    w = (t - mids[j - 1]) / (mids[j] - mids[j - 1])
    return ((1 - w) * table[keys[j - 1]] + w * table[keys[j]]).astype("float32")

def write_like(arr, src, dst, tags):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with rasterio.open(src) as s:
        prof = s.profile; t0 = s.tags()
    prof.update(dtype="float32", nodata=np.nan, count=1)
    t0.update(tags); t0["bias_corrected"] = "yes"
    tmp = dst + ".part"
    try:
        with rasterio.open(tmp, "w", **prof) as o:
            o.write(arr.astype("float32"), 1); o.update_tags(**t0)
        os.replace(tmp, dst)
    except Exception:
        if os.path.exists(tmp): os.remove(tmp)
        raise

def run_county(county):
    G = GRIDS[county]; tabs = {}
    for v in VARS3:
        tabs[v], borrowed = chrono_table(BIAS[county][v], FC_MONTHS)
    for k, src in borrowed:
        print(f"  {county}: {k[0]}-{k[1]:02d} has no bias estimate, using {src[0]}-{src[1]:02d}")
    for stat in STATS:
        for g, v, kind in LAYERS:
            dates = list_dates(county, g, v, stat)
            if not dates:
                print(f"  {county} {stat} {v}: no input files"); continue
            n_new = 0
            for d in dates:
                dst = out_path(county, g, v, stat, d)
                if SKIP_EXISTING and os.path.exists(dst): continue
                src = in_path(county, g, v, stat, d)
                if kind == "copy":                                   # rain and wind_dir: untouched
                    os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(src, dst); n_new += 1; continue
                with rasterio.open(src) as s:
                    a = s.read(1, masked=True).astype("float32").filled(np.nan)
                if a.shape != G["shape"]:
                    raise ValueError(f"{src}: shape {a.shape} differs from the county grid {G['shape']}")
                tags = dict(variable=v, stat=stat)
                if kind == "temp":
                    a = a + at_date(tabs["temp"], d); how = "SEAS5 archive bias, additive"
                elif kind == "tx":
                    if np.isfinite(np.nanmedian(a)) and np.nanmedian(a) > 150: a = a - 273.15          # Kelvin -> degC
                    if APPLY_TO_TXN: a = a + at_date(tabs["temp"], d)
                    how = "SEAS5 archive bias, additive (offset of the mean temperature)" if APPLY_TO_TXN else "unit fix only"
                    tags["units"] = "degC"
                elif kind == "rh":
                    a = np.clip(a + at_date(tabs["rh"], d), 0, 100); how = "SEAS5 archive bias, additive"
                elif kind == "wind":
                    a = a * at_date(tabs["wind"], d); how = "SEAS5 archive bias, ratio"
                write_like(a, src, dst, dict(tags, bias_method=how)); n_new += 1
            print(f"  {county} {stat:4s} {v:10s}: {n_new} written, {len(dates) - n_new} already there", flush=True)

for c in COUNTIES:
    print(f"\n=== {c} ==="); run_county(c)
print("\nALL DONE")

## 6. Check the new folder
- Temperature / RH should shift by about the bias table; wind should scale by about its ratio.
- `mask changed anywhere` must be **False**.
- Rain and wind direction must be **byte-identical** to the originals.

In [ ]:
def rd(p):
    with rasterio.open(p) as s: return s.read(1, masked=True).astype("float32").filled(np.nan)

rows = []
for c in COUNTIES:
    for g, v, kind in LAYERS:
        if kind == "copy": continue
        ds = list_dates(c, g, v, "mean")
        for d in ds[::max(1, len(ds) // 6)]:
            po, pc = in_path(c, g, v, "mean", d), out_path(c, g, v, "mean", d)
            if not (os.path.exists(po) and os.path.exists(pc)): continue
            o, n = rd(po), rd(pc)
            if kind == "tx" and np.nanmedian(o) > 150: o = o - 273.15
            rows.append(dict(county=c, var=v, date=d, original=np.nanmean(o), corrected=np.nanmean(n), diff=np.nanmean(n) - np.nanmean(o),
                             ratio=np.nanmean(n) / np.nanmean(o) if abs(np.nanmean(o)) > 1e-6 else np.nan,
                             mask_same=bool(np.array_equal(np.isfinite(o), np.isfinite(n)))))
chk = pd.DataFrame(rows)
is_ratio = chk["var"] == "wind_speed"
chk["diff"] = chk["diff"].where(~is_ratio); chk["ratio"] = chk["ratio"].where(is_ratio)
print(chk.round(2).to_string(index=False))
print("\nmask changed anywhere:", (~chk.mask_same).any())
print("\nmean change per county and variable (degC / % for temp, tmax, tmin, rh; ratio for wind_speed):")
print(chk.groupby(["county", "var"]).agg(diff=("diff", "mean"), ratio=("ratio", "mean")).round(2).to_string())

print("\nCopied layers (rain, wind_dir): files that differ from the original, or are missing")
for c in COUNTIES:
    for stat in STATS:
        for g, v, kind in LAYERS:
            if kind != "copy": continue
            ds = list_dates(c, g, v, stat)
            bad = [d for d in ds if not (os.path.exists(out_path(c, g, v, stat, d)) and filecmp.cmp(in_path(c, g, v, stat, d), out_path(c, g, v, stat, d), shallow=False))]
            if ds: print(f"  {c} {stat:4s} {v:9s}: {len(ds)} files, {len(bad)} differ/missing")

## 7. What to run next (order matters)
1. **DEM-correction notebook**
2. Run **Notebook A**, **A2** and **C**.
4. Re-run **Notebook A section 8**. Forecast-minus-history offsets for temperature and RH should now be small; rain is deliberately left as the raw forecast.

**Limits.** The bias is estimated from only a few years. It corrects the average error, not the lack of skill of an individual forecast. tmax / tmin carry only the mean-temperature offset.